# GridMET temperature, precipitation and storm detection

Prepares daily GridMET data for the study basins and identifies storms from precipitation. Runs before the storm/climate notebook, which matches these files to the SWE dates. GridMET covers a longer record than the SWE data, so these files do too.

**Steps**
1. **Temperature:** daily mean temperature, (Tmin + Tmax) / 2 [K], cropped to a box around all basins.
2. **Precipitation:** daily precipitation [mm], same box.
3. **Storm labels:** for each pixel, a storm is a run of consecutive wet days that
   - falls inside a 3-day window with more than 200 mm of total precipitation, and
   - totals at least 200 mm itself.

   Each storm is numbered separately at each pixel; the numbers aren't shared across pixels. Days with no storm anywhere are dropped.
4. **Storm days:** the list of days with a storm somewhere in the study area, and in each basin. The per-basin clip needs a larger node.

**Inputs**
- `Data/GridMET/tmmn_*.nc`, `tmmx_*.nc`, `pr_*.nc`: raw GridMET files
- `data_cleaning.py`: basin lists, `shapes`, `data_folder`

**Outputs** (in `Data/Cleaned/GridMET/`)
| File | Contents |
|------|----------|
| `temp_avg_da.nc4` | Daily mean temperature [K] |
| `prec_da.nc4` | Daily precipitation [mm] |
| `storm_labels.nc4` | Storm ID per pixel on storm days (NaN = no storm) |
| `storm_days.nc4` | Days with a storm anywhere in the study area |
| `<basin>_storm_days.nc4` | Days with a storm in that basin |

In [ ]:
# %% Imports and study area
import gc
import glob
import os

import dask
import numpy as np
import rioxarray  # noqa: F401  (registers the .rio accessor)
import xarray as xr
from scipy.ndimage import label as label_runs

from config import data_folder, Basins, basin_names_list, shapes
gridmet_raw = os.path.join(data_folder, 'Data/GridMET')
gridmet_out = os.path.join(data_folder, 'Data/Cleaned/GridMET')
THRESHOLD = 200  # mm: a storm needs a 3-day precipitation total above this

# Bounding box around all study basins
minx, miny, maxx, maxy = shapes[shapes['name'].isin(basin_names_list)].total_bounds


def load_gridmet(prefix, var, new_name):
    """Open every GridMET file for one variable, renamed and cropped to the study area."""
    return [
        xr.open_dataset(f, engine='netcdf4', chunks='auto')
          .rename({var: new_name, 'lon': 'Longitude', 'lat': 'Latitude', 'day': 'Day'})
          .drop_vars('crs')
          .sel(Longitude=slice(minx, maxx), Latitude=slice(maxy, miny))  # GridMET latitude is descending
        for f in glob.glob(os.path.join(gridmet_raw, f'{prefix}_*.nc'))
    ]

In [ ]:
# %% Daily mean temperature [K] = (Tmin + Tmax) / 2
temp = xr.combine_by_coords(
    load_gridmet('tmmn', 'air_temperature', 'Temp_Min') + load_gridmet('tmmx', 'air_temperature', 'Temp_Max'),
    combine_attrs='drop_conflicts')
((temp['Temp_Min'] + temp['Temp_Max']) / 2).to_netcdf(os.path.join(gridmet_out, 'temp_avg_da.nc4'))
del temp
gc.collect()

In [ ]:
# %% Daily precipitation [mm]
prec = xr.combine_by_coords(load_gridmet('pr', 'precipitation_amount', 'Precipitation'),
                            combine_attrs='drop_conflicts')['Precipitation']
dask.compute(prec.to_netcdf(os.path.join(gridmet_out, 'prec_da.nc4'), compute=False), scheduler='threads')
del prec
gc.collect()

In [ ]:
# %% Storm labels
prec = xr.open_dataarray(os.path.join(gridmet_out, 'prec_da.nc4'), chunks='auto').load(scheduler='threads')

# A storm day has precip > 0 and falls inside any 3-day window whose total exceeds THRESHOLD.
# rolling() labels each window by its last day, so the windows containing day i end on i, i+1 or i+2.
wet_window = prec.rolling(Day=3, min_periods=3).sum() > THRESHOLD
in_wet_window = (wet_window
                 | wet_window.shift(Day=-1, fill_value=False)
                 | wet_window.shift(Day=-2, fill_value=False))
storm_mask = (in_wet_window & (prec > 0)).astype(int)

# Number each run of consecutive storm days per pixel (1, 2, 3, …; IDs are per pixel, not shared)
labels = xr.apply_ufunc(lambda m: label_runs(m)[0], storm_mask,
                        input_core_dims=[['Day']], output_core_dims=[['Day']], vectorize=True)
labels = labels.where(labels != 0).dropna('Day', how='all')


def drop_small_storms(precip, labels, threshold):
    """One pixel's time series: remove storms whose total precipitation is below `threshold`."""
    out = labels.copy()
    for storm in np.unique(labels[~np.isnan(labels)]):
        in_storm = labels == storm
        if precip[in_storm].sum() < threshold:
            out[in_storm] = np.nan
    return out


# A zero-precip day can split one wet window into two storms that are each below the threshold
storm_labels = xr.apply_ufunc(
    drop_small_storms, prec.reindex_like(labels), labels, kwargs={'threshold': THRESHOLD},
    input_core_dims=[['Day'], ['Day']], output_core_dims=[['Day']], vectorize=True,
).rename('StormLabel')

storm_labels.to_netcdf(os.path.join(gridmet_out, 'storm_labels.nc4'))
del prec, storm_mask, labels
gc.collect()

In [ ]:
# %% Storm days: whole study area and per basin (per-basin clip needs a bigger node)
storm_labels.dropna('Day', how='all').Day.to_netcdf(os.path.join(gridmet_out, 'storm_days.nc4'))

labels_xy = storm_labels.rename({'Longitude': 'x', 'Latitude': 'y'}).rio.write_crs('EPSG:4326')
for basin in Basins:
    polygon = shapes.loc[shapes['name'] == basin_names[basin], 'geometry'].iloc[0]
    days = labels_xy.rio.clip([polygon], crs='EPSG:4326').dropna('Day', how='all').Day
    days.to_netcdf(os.path.join(gridmet_out, f'{basin}_storm_days.nc4'))